# Uniswap V3 数据下载与策略分析
本 Notebook 用于从 The Graph 与 RPC 下载 Uniswap v3 数据，完成清洗、特征构建、指标计算与策略回测。

## 1. 安装所需依赖（pip）
如需固定版本，可在 requirements.txt 或 Poetry 中维护版本锁。此处记录当前环境版本。

In [ ]:
# %pip install pandas numpy requests web3 pyarrow matplotlib seaborn plotly python-dotenv gql[requests] tqdm

import importlib.metadata as md

packages = [
    "pandas",
    "numpy",
    "requests",
    "web3",
    "pyarrow",
    "matplotlib",
    "seaborn",
    "plotly",
    "python-dotenv",
    "gql",
    "tqdm",
]

versions = {pkg: md.version(pkg) for pkg in packages}
versions

## 2. 配置与环境变量加载
建议在 .env 中配置 GRAPH_API_KEY、SUBGRAPH_ID（或直接提供 GRAPH_ENDPOINT），以及 RPC_URL 等信息。

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

root_dir = Path("..")
load_dotenv(root_dir / "app.env", override=True)
load_dotenv(root_dir / ".env", override=True)

GRAPH_API_KEY = os.getenv("GRAPH_API_KEY", "")
SUBGRAPH_API_BASE = os.getenv("SUBGRAPH_API_BASE", "")
ARBITRUM_SUBGRAPH_ID = os.getenv("ARBITRUM_SUBGRAPH_ID", "")
UNISWAP_V3_ARBITRUM_SUBGRAPH_ID = os.getenv("UNISWAP_V3_ARBITRUM_SUBGRAPH_ID", "")

ARBITRUM_ENDPOINT = f"{SUBGRAPH_API_BASE}/{ARBITRUM_SUBGRAPH_ID}"
UNISWAP_V3_ARBITRUM_ENDPOINT = f"{SUBGRAPH_API_BASE}/{UNISWAP_V3_ARBITRUM_SUBGRAPH_ID}"

ARBITRUM_ENDPOINT, UNISWAP_V3_ARBITRUM_ENDPOINT

In [ ]:
ALCHEMY_API_KEY = os.getenv("ALCHEMY_API_KEY", "")
RPC_URL = os.getenv("ALCHEMY_ARBITRUM_URL", "")

RPC_URL if RPC_URL else ""

## 3. 连接并验证数据源（The Graph / RPC）
验证 GraphQL 与 RPC 是否可用。

In [ ]:
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport

headers = {"Authorization": f"Bearer {GRAPH_API_KEY}"} if GRAPH_API_KEY else {}
transport = RequestsHTTPTransport(url=ARBITRUM_ENDPOINT, verify=True, retries=3, headers=headers)
gql_client = Client(transport=transport, fetch_schema_from_transport=False)

query = gql("""
{
  _meta {
    block { number }
  }
}
""")

meta = gql_client.execute(query)
meta

In [ ]:
from web3 import Web3

# Add api key to headers
headers = {"Authorization": f"Bearer {ALCHEMY_API_KEY}"} if ALCHEMY_API_KEY else {}
w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={"headers": headers}))

if RPC_URL:
    assert w3.is_connected(), "RPC connection failed"
    chain_id = w3.eth.chain_id
else:
    chain_id = None

chain_id

## 4. 下载 Uniswap v3 池子与交易数据
按池子地址与时间范围拉取 swaps、ticks、liquidity events 并缓存。

In [ ]:
import pandas as pd
from datetime import datetime, timezone
from tqdm import tqdm

DATA_DIR = (root_dir / "data" / "raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

POOL_ADDRESS = os.getenv("POOL_ADDRESS", "0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640").lower()  # USDC/WETH 0.05%
START_TS = int(os.getenv("START_TS", "1704067200"))  # 2024-01-01
END_TS = int(os.getenv("END_TS", "1706745600"))    # 2024-02-01

PAGE_SIZE = 1000

SWAPS_QUERY = gql("""
query($pool: String!, $ts: Int!, $pageSize: Int!) {
  swaps(first: $pageSize, orderBy: timestamp, orderDirection: asc, where: { pool: $pool, timestamp_gte: $ts }) {
    id
    timestamp
    sqrtPriceX96
    amount0
    amount1
    amountUSD
    tick
    liquidity
  }
}
""")

TICKS_QUERY = gql("""
query($pool: String!, $ts: Int!, $pageSize: Int!) {
  ticks(first: $pageSize, orderBy: createdAtTimestamp, orderDirection: asc, where: { pool: $pool, createdAtTimestamp_gte: $ts }) {
    id
    tickIdx
    liquidityGross
    liquidityNet
    createdAtTimestamp
  }
}
""")

LIQ_EVENTS_QUERY = gql("""
query($pool: String!, $ts: Int!, $pageSize: Int!) {
  mints(first: $pageSize, orderBy: timestamp, orderDirection: asc, where: { pool: $pool, timestamp_gte: $ts }) {
    id
    timestamp
    amount
    amount0
    amount1
    owner
    tickLower
    tickUpper
  }
  burns(first: $pageSize, orderBy: timestamp, orderDirection: asc, where: { pool: $pool, timestamp_gte: $ts }) {
    id
    timestamp
    amount
    amount0
    amount1
    owner
    tickLower
    tickUpper
  }
}
""")


def paginate(query, extract_key, variables, end_ts):
    rows = []
    ts_cursor = variables["ts"]
    while ts_cursor < end_ts:
        variables["ts"] = ts_cursor
        data = gql_client.execute(query, variable_values=variables)
        batch = data.get(extract_key, [])
        if not batch:
            break
        rows.extend(batch)
        last_ts = int(batch[-1]["timestamp"] if "timestamp" in batch[-1] else batch[-1].get("createdAtTimestamp", ts_cursor))
        ts_cursor = last_ts + 1
        if len(batch) < variables["pageSize"]:
            break
    return rows

swaps = paginate(SWAPS_QUERY, "swaps", {"pool": POOL_ADDRESS, "ts": START_TS, "pageSize": PAGE_SIZE}, END_TS)
ticks = paginate(TICKS_QUERY, "ticks", {"pool": POOL_ADDRESS, "ts": START_TS, "pageSize": PAGE_SIZE}, END_TS)
liq_events = gql_client.execute(LIQ_EVENTS_QUERY, variable_values={"pool": POOL_ADDRESS, "ts": START_TS, "pageSize": PAGE_SIZE})

swaps_df = pd.DataFrame(swaps)
ticks_df = pd.DataFrame(ticks)
mints_df = pd.DataFrame(liq_events.get("mints", []))
burns_df = pd.DataFrame(liq_events.get("burns", []))

swaps_df.to_parquet(DATA_DIR / f"swaps_{POOL_ADDRESS}_{START_TS}_{END_TS}.parquet", index=False)
ticks_df.to_parquet(DATA_DIR / f"ticks_{POOL_ADDRESS}_{START_TS}_{END_TS}.parquet", index=False)
mints_df.to_parquet(DATA_DIR / f"mints_{POOL_ADDRESS}_{START_TS}_{END_TS}.parquet", index=False)
burns_df.to_parquet(DATA_DIR / f"burns_{POOL_ADDRESS}_{START_TS}_{END_TS}.parquet", index=False)

len(swaps_df), len(ticks_df), len(mints_df), len(burns_df)

## 5. 数据清洗与特征构建
统一时间索引与特征工程。

In [ ]:
import numpy as np

swaps_df["timestamp"] = swaps_df["timestamp"].astype(int)
swaps_df["datetime"] = pd.to_datetime(swaps_df["timestamp"], unit="s", utc=True)

for col in ["amount0", "amount1", "amountUSD", "liquidity", "sqrtPriceX96", "tick"]:
    if col in swaps_df.columns:
        swaps_df[col] = pd.to_numeric(swaps_df[col], errors="coerce")

swaps_df = swaps_df.dropna(subset=["amountUSD", "sqrtPriceX96"]).sort_values("datetime")

# 价格特征：将 sqrtPriceX96 转为价格（需按 token0/token1 方向做修正）
swaps_df["price"] = (swaps_df["sqrtPriceX96"] / 2**96) ** 2
swaps_df["log_price"] = np.log(swaps_df["price"])

# 成交量与方向
swaps_df["abs_amountUSD"] = swaps_df["amountUSD"].abs()
swaps_df["direction"] = np.sign(swaps_df["amount1"])

swaps_df.head()

## 6. 计算流动性与价格指标
计算流动性、TWAP 等指标。

In [ ]:
# TWAP（示例：5分钟窗口）
window = "5min"

swaps_df = swaps_df.set_index("datetime")
swaps_df["twap"] = swaps_df["log_price"].rolling(window).mean().pipe(np.exp)

# 流动性统计
swaps_df["liquidity"] = swaps_df["liquidity"].fillna(method="ffill")
liquidity_stats = swaps_df["liquidity"].describe()

liquidity_stats

## 7. 费用与滑点估算
基于历史成交估算费用与冲击成本。

In [ ]:
FEE_TIER = float(os.getenv("FEE_TIER", "0.0005"))  # 0.05%

# 费用估算：成交额 * 费率
swaps_df["fee_usd"] = swaps_df["abs_amountUSD"] * FEE_TIER

# 简化滑点估计：交易额 / 流动性
swaps_df["slippage_proxy"] = swaps_df["abs_amountUSD"] / swaps_df["liquidity"].replace(0, np.nan)

swaps_df[["fee_usd", "slippage_proxy"]].describe()

## 8. 回测策略逻辑实现
示例策略：基于 TWAP 的均值回归信号。

In [ ]:
# 简单信号：价格偏离 TWAP
swaps_df["signal"] = (swaps_df["price"] - swaps_df["twap"]) / swaps_df["twap"]

# 仓位：偏离过大则反向
threshold = float(os.getenv("SIGNAL_THRESHOLD", "0.002"))
swaps_df["position"] = np.where(swaps_df["signal"] > threshold, -1, np.where(swaps_df["signal"] < -threshold, 1, 0))

# 简化收益：下一期价格变动 * 仓位
swaps_df["price_return"] = swaps_df["price"].pct_change().shift(-1)
swaps_df["strategy_return"] = swaps_df["position"] * swaps_df["price_return"]

swaps_df[["strategy_return"]].dropna().describe()

## 9. 结果可视化与评估
输出收益曲线、回撤与风险指标。

In [ ]:
import matplotlib.pyplot as plt

result = swaps_df[["strategy_return"]].dropna().copy()
result["equity"] = (1 + result["strategy_return"]).cumprod()
result["peak"] = result["equity"].cummax()
result["drawdown"] = result["equity"] / result["peak"] - 1

fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
result["equity"].plot(ax=ax[0], title="Equity Curve")
result["drawdown"].plot(ax=ax[1], title="Drawdown")
plt.tight_layout()

result[["equity", "drawdown"]].tail()